# Introduction

This notebook demonstrates an ID3 Decision Tree workflow. It loads small categorical datasets, implements entropy and information gain, builds custom ID3 trees, evaluates predictions, runs cross-validation, discretizes Iris features when sklearn is available, saves readable tree outputs, compares against sklearn, and demonstrates reduced error pruning.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import discretize, load_iris_dataset, load_sunburn, load_tennis
from src.evaluation import compare_with_sklearn, compute_accuracy, cross_validate, predict
from src.id3 import entropy, id3, information_gain, majority_class, tree_size
from src.pruning import pruning_experiment
from src.visualization import print_tree, save_text_tree

## Dataset Loading

Load the Tennis and Sunburn categorical datasets. Iris is loaded from sklearn when the dependency is installed; otherwise, the notebook continues with a clear skip message.

In [ ]:
tennis_df = load_tennis()
sunburn_df = load_sunburn()

try:
    iris_df = load_iris_dataset()
except ImportError as error:
    iris_df = None
    print(f"Iris dataset unavailable: {error}")

print(f"Tennis dataset loaded: rows={tennis_df.shape[0]}, columns={tennis_df.shape[1]}")
print(f"Sunburn dataset loaded: rows={sunburn_df.shape[0]}, columns={sunburn_df.shape[1]}")
if iris_df is not None:
    print(f"Iris dataset loaded: rows={iris_df.shape[0]}, columns={iris_df.shape[1]}")
else:
    print("Iris dataset loaded: unavailable")

In [ ]:
display(tennis_df.head())
display(sunburn_df.head())
if iris_df is not None:
    display(iris_df.head())

## Entropy and Information Gain

Verify entropy, majority class, and information gain before building the recursive tree.

In [ ]:
entropy_examples = [
    ['yes', 'yes', 'no', 'yes'],
    ['yes', 'yes', 'yes'],
    ['yes', 'yes', 'no', 'no'],
]

for example in entropy_examples:
    print(f"Entropy({example}): {entropy(example):.6f}")

print(f"Entropy example: {entropy(['yes', 'yes', 'no', 'yes']):.6f}")
print(f"Majority class example: {majority_class(['yes', 'yes', 'no', 'no'])}")

In [ ]:
tennis_target = 'play' if 'play' in tennis_df.columns else 'Play'
tennis_features = [column for column in tennis_df.columns if column != tennis_target]

for feature in tennis_features:
    gain = information_gain(tennis_df, feature, tennis_target)
    print(f"Gain({feature.title()}): {gain:.4f}")

sunburn_target = 'Burnt'
sunburn_features = [
    column for column in sunburn_df.columns if column not in ['Name', sunburn_target]
]

for feature in sunburn_features:
    gain = information_gain(sunburn_df, feature, sunburn_target)
    print(f"Gain({feature}): {gain:.4f}")

## Core ID3 Tree Builder

Train custom nested-dictionary ID3 trees on Tennis and Sunburn. Prediction and evaluation are handled in later sections.

In [ ]:
tennis_tree = id3(tennis_df, tennis_features, tennis_target)
sunburn_tree = id3(sunburn_df, sunburn_features, sunburn_target)

print("Tennis tree:")
print(tennis_tree)
print(f"Tennis tree size: {tree_size(tennis_tree)}")

print("\nSunburn tree:")
print(sunburn_tree)
print(f"Sunburn tree size: {tree_size(sunburn_tree)}")

## Prediction and Evaluation

Use the trained custom trees for single-row predictions and full-dataset training accuracy.

In [ ]:
tennis_examples = [
    {'outlook': 'Sunny', 'temperature': 'Hot', 'humidity': 'High', 'wind': 'Weak'},
    {'outlook': 'Overcast', 'temperature': 'Cool', 'humidity': 'Normal', 'wind': 'Strong'},
    {'outlook': 'Rain', 'temperature': 'Mild', 'humidity': 'High', 'wind': 'Strong'},
]

for example in tennis_examples:
    prediction = predict(tennis_tree, example, default='Unknown')
    print(f"Tennis prediction for {example}: {prediction}")

tennis_accuracy = compute_accuracy(tennis_tree, tennis_df, tennis_target, default='Unknown')
print(f"Tennis training accuracy: {tennis_accuracy:.4f}")

In [ ]:
sunburn_examples = [
    {'Hair': 'Blonde', 'Height': 'Short', 'Weight': 'Light', 'Lotion': 'No'},
    {'Hair': 'Brown', 'Height': 'Tall', 'Weight': 'Heavy', 'Lotion': 'No'},
    {'Hair': 'Red', 'Height': 'Average', 'Weight': 'Heavy', 'Lotion': 'Yes'},
]

for example in sunburn_examples:
    prediction = predict(sunburn_tree, example, default='Unknown')
    print(f"Sunburn prediction for {example}: {prediction}")

sunburn_accuracy = compute_accuracy(sunburn_tree, sunburn_df, sunburn_target, default='Unknown')
print(f"Sunburn training accuracy: {sunburn_accuracy:.4f}")

## Cross-Validation

Run shuffled k-fold cross-validation with the custom ID3 implementation. Tennis uses 5 folds, and Sunburn uses 4 folds because it has only 8 rows.

In [ ]:
tennis_cv = cross_validate(tennis_df, tennis_features, tennis_target, k=5, random_state=42)
sunburn_cv = cross_validate(sunburn_df, sunburn_features, sunburn_target, k=4, random_state=42)

print(f"Tennis fold accuracies: {tennis_cv['fold_results']}")
print(f"Tennis mean accuracy: {tennis_cv['mean_accuracy']:.4f}")
print(f"Tennis std accuracy: {tennis_cv['std_accuracy']:.4f}")

print(f"\nSunburn fold accuracies: {sunburn_cv['fold_results']}")
print(f"Sunburn mean accuracy: {sunburn_cv['mean_accuracy']:.4f}")
print(f"Sunburn std accuracy: {sunburn_cv['std_accuracy']:.4f}")

## Iris Discretization

Convert numerical Iris attributes into equal-frequency categorical bins so the custom ID3 tree can train on them. This section is skipped when sklearn is unavailable.

In [ ]:
iris_attributes = [
    'sepal length (cm)',
    'sepal width (cm)',
    'petal length (cm)',
    'petal width (cm)',
]

if iris_df is not None:
    print("Iris before discretization:")
    display(iris_df.head())

    iris_discretized = discretize(iris_df, iris_attributes, num_bins=3)

    print("Iris after discretization:")
    display(iris_discretized.head())

    iris_tree = id3(iris_discretized, iris_attributes, 'target_name')
    iris_cv = cross_validate(iris_discretized, iris_attributes, 'target_name', k=5, random_state=42)

    print(f"Iris tree size: {tree_size(iris_tree)}")
    print(f"Iris fold accuracies: {iris_cv['fold_results']}")
    print(f"Iris mean accuracy: {iris_cv['mean_accuracy']:.4f}")
    print(f"Iris std accuracy: {iris_cv['std_accuracy']:.4f}")
else:
    print("Skipping Iris discretization because sklearn is unavailable. Run pip install -r requirements.txt to enable this section.")

## Tree Visualization

Format custom ID3 trees as readable text and save them under `outputs/figures/`.

In [ ]:
figures_dir = PROJECT_ROOT / 'outputs' / 'figures'

print("Formatted Tennis tree:")
print_tree(tennis_tree)
save_text_tree(tennis_tree, figures_dir / 'tennis_tree.txt')

print("\nFormatted Sunburn tree:")
print_tree(sunburn_tree)
save_text_tree(sunburn_tree, figures_dir / 'sunburn_tree.txt')

if 'iris_tree' in globals():
    print("\nFormatted Iris tree:")
    print_tree(iris_tree)
    save_text_tree(iris_tree, figures_dir / 'iris_tree.txt')

print(f"Saved text trees to {figures_dir}")

## Comparison with sklearn

Compare the custom ID3 implementation with sklearn's entropy-based `DecisionTreeClassifier`. If sklearn is unavailable, save an empty comparison table with the expected columns and print an install message.

In [ ]:
comparison_columns = ['Dataset', 'Algorithm', 'Accuracy', 'Tree Size', 'Train Time (ms)']
comparison_tables = []

try:
    comparison_tables.append(compare_with_sklearn('Tennis', tennis_df, tennis_features, tennis_target))
    comparison_tables.append(compare_with_sklearn('Sunburn', sunburn_df, sunburn_features, sunburn_target))

    if 'iris_discretized' in globals():
        comparison_tables.append(compare_with_sklearn('Iris', iris_discretized, iris_attributes, 'target_name'))
    else:
        print("Skipping Iris sklearn comparison because Iris data is unavailable.")

    comparison_results = pd.concat(comparison_tables, ignore_index=True)
except ImportError as error:
    print(error)
    comparison_results = pd.DataFrame(columns=comparison_columns)

comparison_output_path = PROJECT_ROOT / 'outputs' / 'tables' / 'comparison_results.csv'
comparison_output_path.parent.mkdir(parents=True, exist_ok=True)
comparison_results.to_csv(comparison_output_path, index=False)

display(comparison_results)
print(f"Saved comparison table to {comparison_output_path}")

## Reduced Error Pruning

Run a reduced error pruning experiment. Iris is preferred when available; otherwise, the notebook falls back to Tennis so the section can still run without sklearn.

In [13]:
if 'iris_discretized' in globals():
    pruning_results = pruning_experiment(iris_discretized, iris_attributes, 'target_name', random_state=42)
else:
    print("Running pruning fallback on Tennis because Iris is unavailable.")
    pruning_results = pruning_experiment(tennis_df, tennis_features, tennis_target, random_state=42)

pruning_output_path = PROJECT_ROOT / 'outputs' / 'tables' / 'pruning_results.csv'
pruning_output_path.parent.mkdir(parents=True, exist_ok=True)
pruning_results.to_csv(pruning_output_path, index=False)

display(pruning_results)
print(f"Saved pruning results to {pruning_output_path}")

,Metric,Unpruned,Pruned
0,Training Accuracy,0.990566,0.981132
1,Validation Accuracy,0.909091,1.000000
2,Test Accuracy,0.954545,0.954545
3,Number of Nodes,13.000000,6.000000


Saved pruning results to C:\Users\David M. Gegbai\Documents\Codex Projects\ID3\outputs\tables\pruning_results.csv


## Result Summary

The notebook now demonstrates the full custom ID3 workflow from dataset loading through pruning. The core ID3 calculations, tree construction, prediction, cross-validation, text visualization, and pruning are implemented from scratch. sklearn is used only for Iris loading and comparison against `DecisionTreeClassifier` when dependencies are installed.

Expected saved outputs:

- `outputs/figures/tennis_tree.txt`
- `outputs/figures/sunburn_tree.txt`
- `outputs/tables/comparison_results.csv`
- `outputs/tables/pruning_results.csv`